# Create Icechunk from Kerchunk

In [21]:
# get kerchunks
import gcsfs

fs = gcsfs.GCSFileSystem(token="anon")

kerchunk_jsons = fs.glob(
    "noaa-oar-cefi-regional-mom6/"
    "northwest_atlantic/full_domain/hindcast/monthly/regrid/"
    "r20250715/*.json"
)
kerchunk_jsons = [
    f if f.startswith("gcs://") else "https://storage.googleapis.com/" + f
    for f in files
    if f.endswith(".json")
]
for f in kerchunk_jsons[0:3]:
    print(f)

# https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json

https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/BHEAT.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/BMELT.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json


In [20]:
import obstore.store as obs

from virtualizarr import open_virtual_dataset
from virtualizarr.registry import ObjectStoreRegistry
from virtualizarr.parsers import KerchunkJSONParser

registry = ObjectStoreRegistry()

store = obs.HTTPStore(
    "https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/"
)

registry.register(
    "https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/",
    store,
)

vsets = [
    open_virtual_dataset(
        ref,
        registry=registry,
        parser=KerchunkJSONParser(),
    )
    for ref in kerchunk_jsons
]

FileNotFoundError: Object at location northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json not found: Error performing GET https://storage.googleapis.com/noaa-oar-cefi-regional-mom6//northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json in 67.781845ms - Server returned non-2xx status code: 404 Not Found: <?xml version='1.0' encoding='UTF-8'?><Error><Code>NoSuchKey</Code><Message>The specified key does not exist.</Message><Details>No such object: noaa-oar-cefi-regional-mom6//northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json</Details></Error>

Debug source:
NotFound {
    path: "northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json",
    source: RetryError(
        RetryErrorImpl {
          

In [24]:
import obstore.store as obs

from virtualizarr import open_virtual_dataset
from virtualizarr.registry import ObjectStoreRegistry
from virtualizarr.parsers import KerchunkJSONParser

BASE = "https://storage.googleapis.com/noaa-oar-cefi-regional-mom6"

kerchunk_jsons_https = [
    f.replace(
        "gcs://noaa-oar-cefi-regional-mom6",
        BASE,
    )
    for f in kerchunk_jsons
]

registry = ObjectStoreRegistry()

store = obs.HTTPStore(BASE)

registry.register(BASE, store)

vsets = [
    open_virtual_dataset(
        ref,
        registry=registry,
        parser=KerchunkJSONParser(),
    )
    for ref in kerchunk_jsons_https
]


ValueError: paths in the manifest must be absolute posix paths or URIs, but got gcs://noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.nc, and fs_root was not specified

In [23]:
print(kerchunk_jsons_https[0])

https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json


In [ ]:


# Combine them, if needed
vds = xr.combine_by_coords(
    vsets,
    combine_attrs="override",
)

# Create Icechunk output repo
storage = icechunk.s3_storage(
    bucket="YOUR_OUTPUT_BUCKET",
    prefix="path/to/output.icechunk",
    region="us-west-2",
    from_env=True,
)

repo = icechunk.Repository.create(storage)

# Open repo authorizing the original public GCS NetCDF source
# The refs inside the kerchunk JSON point here.
virtual_chunk_container = icechunk.gcs_storage(
    bucket="noaa-oar-cefi-regional-mom6",
    prefix="",
    anonymous=True,
)

repo = icechunk.Repository.open(
    storage,
    authorize_virtual_chunk_access=[virtual_chunk_container],
)

session = repo.writable_session("main")

# Write virtual references into Icechunk
vds.virtualize.to_icechunk(session.store)

session.commit("Create Icechunk store from existing Kerchunk JSON refs")